# Playground

This script create from an excel file a long table in a csv file.


## (1) Reading Excel

In [4]:
import pandas as pd

# Reading excel file and name of the sheet as data frame 
df = pd.read_excel(r"N:\C2603_LandUse_2024\f04_tableau\f07_tabular_data_source\data_from_gis\landuse2.xlsx", sheet_name='change')

# Display the dataframe
print(df.head())

  LU1_2024 LU1_2021 LU1_2018 LU1_2015 LU1_2007  Sum of area_ha change24_21
0      111      111      111      111      111    11026.239155   no change
1      111      111      111      111      112        0.545213   no change
2      111      111      111      111      113        2.339550   no change
3      111      111      111      111      114       89.550341   no change
4      111      111      111      111      115        0.657609   no change


In [9]:
# Transform data frame into long format, using melt function. Creating new column storing every year with landuse class for each polygon.
long_df = df.melt(
    id_vars=["Sum of area_ha", "change24_21"],  # keep these columns 
    var_name="YEAR",  # New column for the year
    value_name="LU1"  # New column for the class values
)

# Extract the year from the year classification column (e.g., 'LU1_2024' -> '2024')
long_df["YEAR"] = long_df["YEAR"].str.extract(r'(\d{4})')

# Display the long DataFrame
print(long_df)

       Sum of area_ha change24_21  YEAR      LU1
0        11026.239155   no change  2024      111
1            0.545213   no change  2024      111
2            2.339550   no change  2024      111
3           89.550341   no change  2024      111
4            0.657609   no change  2024      111
...               ...         ...   ...      ...
20955        0.304217   no change  2007      612
20956        0.059173   no change  2007      620
20957      272.930366   no change  2007      650
20958             NaN   no change  2007  (blank)
20959   259306.521751      change  2007      NaN

[20960 rows x 4 columns]


In [11]:
from tabulate import tabulate

# Convert DataFrame to a formatted table
table = tabulate(long_df, headers='keys', tablefmt='psql', showindex=False)
#print(table)

In [12]:


# Save the long DataFrame as a CSV file
long_df.to_csv(r"N:\C2603_LandUse_2024\f04_tableau\f07_tabular_data_source\data_from_gis\export_from_py\long_table.csv", index=False)

## (2) Export features form ArcGIS

Exporting master gdb with communes names in order to filter in tableau 
--------- Use arcgis pro kernel to run this script. ---------

In [2]:
import arcpy

# gdb file
arcpy.env.workspace = r"D:\landuse\statistics\landuse2024.gdb"

# List all feature classes or tables
feature_classes = arcpy.ListFeatureClasses()
tables = arcpy.ListTables()

# Export each feature class or table to CSV
for fc in feature_classes:
    arcpy.TableToTable_conversion(fc, r"N:\C2603_LandUse_2024\f04_tableau\f07_tabular_data_source\data_from_gis", f"{fc}.csv")

for table in tables:
    arcpy.TableToTable_conversion(table, r"N:\C2603_LandUse_2024\f04_tableau\f07_tabular_data_source\data_from_gis", f"{table}.csv")

## (3) Process level 3 data in order to create level 1 sankey diagram in tableau

In [5]:
df2 = pd.read_csv(r"N:\C2603_LandUse_2024\f04_tableau\f07_tabular_data_source\data_from_gis\landuse_with_sheets_change.csv")

print(df2.head())

  LU1_2024 LU1_2021 LU1_2018 LU1_2015 LU1_2007  Sum of area_ha change24_21
0      111      111      111      111      111    11026.239160   no change
1      111      111      111      111      112        0.545213   no change
2      111      111      111      111      113        2.339550   no change
3      111      111      111      111      114       89.550341   no change
4      111      111      111      111      115        0.657609   no change


In [7]:
# Extract Level 1 codes from Level 3 codes
df2["Level1_2021"] = df2["LU1_2021"].astype(str).str[0]  # First digit
df2["Level1_2024"] = df2["LU1_2024"].astype(str).str[0]  # First digit

# Add a column to indicate "change" or "no change"
df2["Change_Status"] = df2.apply(
    lambda row: "no change" if row["Level1_2021"] == row["Level1_2024"] else "change",
    axis=1
)

# Select only the Level 1 columns, the area, and the status
sankey_df2 = df2[["Level1_2021", "Level1_2024", "Sum of area_ha", "Change_Status"]]

# # Group by source (Level 1 in 2021), target (Level 1 in 2024), and change status, and sum the area
# sankey_data = sankey_df2.groupby(["Level1_2021", "Level1_2024", "Change_Status"]).sum().reset_index()

# Save to CSV for Tableau
sankey_df2.to_csv(r"N:\C2603_LandUse_2024\f04_tableau\f07_tabular_data_source\data_from_gis\sankey_level1_transitions.csv", index=False)

update Lau2 column in the commune shp for matching LAU2 column in gdb 
--------- Use arcgis pro kernel to run this script. ---------

In [2]:
import arcpy

table_path =r"N:\C2603_LandUse_2024\f04_tableau\f06_spatial_data_source\communes_lux_3857.dbf"

field_name = "Lau2"

with arcpy.da.UpdateCursor(table_path, [field_name]) as cursor :
    for row in cursor :
        val = row[0]
        if val and val.startswith("0") :
            row[0] = val[1:]
            cursor.updateRow(row)

with arcpy.da.SearchCursor(table_path, [field_name]) as cucrsor :
    for row in cursor :
        print(row[0])

704
1107
303
901
407
213
205
310
307
802
805
701
208
504
707
609
1203
105
502
605
206
608
1004
202
302
606
1103
408
410
110
505
1207
702
108
709
308
406
1002
201
106
311
804
1001
409
402
401
604
706
209
102
104
210
405
710
1202
603
403
610
103
602
705
1201
203
1106
1101
1104
501
214
607
903
703
801
207
1105
1208
1005
902
1102
806
1008
807
708
309
107
1006
204
212
301
1204
808
211
306
1003
503
1206
101
1108
601
305
304
1205
404
105
106
308
308
304
304
305
